# Short

In [1]:
import json
from pathlib import Path

path = Path("timebound_llm_outputs/oracle_evidence/predictions.jsonl")

bad = []
good = []

with path.open("r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        if obj["answer_correct"]:
            good.append(obj)
        else:
            bad.append(obj)

print("good:", len(good))
print("bad:", len(bad))
print("bad rate:", len(bad) / (len(good) + len(bad)))

for x in bad[:30]:
    print("=" * 120)
    print("ID:", x["id"])
    print("TASK:", x["task_type"])
    print("DIFFICULTY:", x["difficulty"])
    print("QUERY:", x["query"])
    print("GOLD:", x["gold_answer"])
    print("PRED:", x["predicted_answer"])
    print("GOLD EVID:", x["gold_evidence_turns"])
    print("PRED EVID:", x["predicted_evidence_turns"])
    print("REASON:", x["reasoning_brief"])

good: 440
bad: 390
bad rate: 0.46987951807228917
ID: timebound_openai_000007
TASK: elapsed_time_reasoning
DIFFICULTY: medium
QUERY: How many hours elapsed since landing at 2026-12-20T20:00:00?
GOLD: 56 hours
PRED: 48
GOLD EVID: [0, 1]
PRED EVID: [0]
REASON: The landing occurred at 2026-12-18T12:00, and the current time is 2026-12-20T20:00, so 2 days and 8 hours (48 hours) have elapsed.
ID: timebound_openai_000008
TASK: rescheduling
DIFFICULTY: hard
QUERY: What are the valid maintenance window times at 2027-01-10T10:00:00?
GOLD: From 2027-01-15T03:00:00 to 2027-01-15T07:00:00
PRED: The valid maintenance window on 2027-01-15 is from 03:00 to 07:00.
GOLD EVID: [0, 1, 2]
PRED EVID: [0, 1]
REASON: The original maintenance window was scheduled from 01:00 to 05:00 on 2027-01-15 (turn 0), but it was rescheduled to 03:00 to 07:00 on the same day (turn 1). The latest valid update at the current time 2027-01-10T10:00:00 is the rescheduled window.
ID: timebound_openai_000010
TASK: periodic_event
D

In [5]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
from dateutil import parser as dtparser

import pandas as pd


# ============================================================
# Config
# ============================================================

OUTPUT_DIR = Path("timebound_llm_outputs")

MODES = [
    "full_history",
    "semantic_rag",
    "timebound_rag",
    "oracle_evidence",
]

OUT_ANALYSIS = OUTPUT_DIR / "failure_analysis"
OUT_ANALYSIS.mkdir(parents=True, exist_ok=True)


# ============================================================
# Normalization utilities
# ============================================================

MONTHS = {
    "january": "01",
    "february": "02",
    "march": "03",
    "april": "04",
    "may": "05",
    "june": "06",
    "july": "07",
    "august": "08",
    "september": "09",
    "october": "10",
    "november": "11",
    "december": "12",
}


def norm_text(s: str) -> str:
    s = str(s).lower().strip()

    s = s.replace("a.m.", "am")
    s = s.replace("p.m.", "pm")
    s = s.replace("a.m", "am")
    s = s.replace("p.m", "pm")

    s = s.replace("at ", " ")
    s = s.replace("on ", " ")
    s = re.sub(r"\s+", " ", s)
    s = s.strip(" .,:;!?")

    return s


def extract_numbers(s: str):
    return re.findall(r"\d+", str(s))


def extract_yes_no(s: str):
    s = norm_text(s)

    if s.startswith("yes") or s in {"true", "valid", "active"}:
        return "yes"

    if (
        s.startswith("no")
        or "not valid" in s
        or "not active" in s
        or "cancelled" in s
        or "canceled" in s
        or "expired" in s
    ):
        return "no"

    return None


def try_parse_datetime_fragments(s: str):
    """
    Extracts dates/datetimes from natural language and ISO-like strings.
    Returns normalized strings:
      YYYY-MM-DD
      YYYY-MM-DDTHH:MM
    """
    s0 = str(s)
    found = set()

    # ISO date/datetime
    iso_patterns = re.findall(
        r"\d{4}-\d{2}-\d{2}(?:[T\s]\d{2}:\d{2}(?::\d{2})?)?",
        s0,
    )

    for x in iso_patterns:
        try:
            dt = dtparser.parse(x)
            if re.search(r"\d{2}:\d{2}", x):
                found.add(dt.strftime("%Y-%m-%dT%H:%M"))
            else:
                found.add(dt.strftime("%Y-%m-%d"))
        except Exception:
            pass

    # Natural date: April 22, 2026 / May 1, 2026
    natural_patterns = re.findall(
        r"(january|february|march|april|may|june|july|august|september|october|november|december)\s+"
        r"(\d{1,2})(?:st|nd|rd|th)?(?:,)?\s+(\d{4})",
        s0,
        flags=re.I,
    )

    for month, day, year in natural_patterns:
        mm = MONTHS[month.lower()]
        dd = str(day).zfill(2)
        found.add(f"{year}-{mm}-{dd}")

    return found


def extract_times(s: str):
    """
    Extracts coarse time mentions:
      14:00
      2 PM -> 14:00
      10 AM -> 10:00
    """
    s0 = str(s).lower()
    times = set()

    # HH:MM
    for h, m in re.findall(r"\b(\d{1,2}):(\d{2})\b", s0):
        hh = int(h)
        mm = int(m)
        if 0 <= hh <= 23 and 0 <= mm <= 59:
            times.add(f"{hh:02d}:{mm:02d}")

    # 2 pm / 10 AM
    for h, ampm in re.findall(r"\b(\d{1,2})\s*(am|pm)\b", s0):
        hh = int(h)
        if ampm == "pm" and hh != 12:
            hh += 12
        if ampm == "am" and hh == 12:
            hh = 0
        if 0 <= hh <= 23:
            times.add(f"{hh:02d}:00")

    return times


def normalize_duration_hours(s: str):
    """
    Extract duration in hours from:
      56 hours
      5 days and 21 hours
      2 days 8 hours
    """
    s0 = norm_text(s)

    # Direct hours
    m = re.search(r"\b(\d+)\s*hours?\b", s0)
    direct_hours = int(m.group(1)) if m else None

    # days + hours
    d = re.search(r"\b(\d+)\s*days?\b", s0)
    h = re.search(r"\b(\d+)\s*hours?\b", s0)

    if d:
        days = int(d.group(1))
        hours = int(h.group(1)) if h else 0
        return days * 24 + hours

    return direct_hours


def relaxed_match(pred: str, gold: str) -> bool:
    p = norm_text(pred)
    g = norm_text(gold)

    if p == g:
        return True

    if p in g or g in p:
        return True

    # yes/no
    py = extract_yes_no(pred)
    gy = extract_yes_no(gold)

    if py is not None and gy is not None and py == gy:
        return True

    # dates
    p_dates = try_parse_datetime_fragments(pred)
    g_dates = try_parse_datetime_fragments(gold)

    if g_dates and p_dates:
        # all gold dates/datetimes appear in prediction, allowing date-only matching
        for gd in g_dates:
            gd_date = gd.split("T")[0]
            if not any(pd == gd or pd.split("T")[0] == gd_date for pd in p_dates):
                break
        else:
            return True

    # times
    p_times = extract_times(pred)
    g_times = extract_times(gold)

    if g_times and p_times and g_times.issubset(p_times):
        return True

    # duration hours
    ph = normalize_duration_hours(pred)
    gh = normalize_duration_hours(gold)

    if ph is not None and gh is not None and ph == gh:
        return True

    # numeric containment
    pn = extract_numbers(pred)
    gn = extract_numbers(gold)

    if gn and pn:
        if all(x in pn for x in gn):
            return True

    return False


# ============================================================
# Failure categorization
# ============================================================

def categorize_failure(pred: str, gold: str, row: dict) -> str:
    p = norm_text(pred)
    g = norm_text(gold)

    if relaxed_match(pred, gold):
        return "false_negative_format_or_paraphrase"

    py = extract_yes_no(pred)
    gy = extract_yes_no(gold)

    if py is not None or gy is not None:
        if py != gy:
            return "yes_no_semantic_error"
        return "yes_no_format_issue"

    ph = normalize_duration_hours(pred)
    gh = normalize_duration_hours(gold)

    if gh is not None:
        if ph is None:
            return "duration_missing_or_unparsed"
        if ph != gh:
            return "duration_arithmetic_error"

    p_dates = try_parse_datetime_fragments(pred)
    g_dates = try_parse_datetime_fragments(gold)

    if g_dates:
        if not p_dates:
            return "date_missing_or_unparsed"
        if p_dates != g_dates:
            return "date_or_calendar_error"

    p_times = extract_times(pred)
    g_times = extract_times(gold)

    if g_times:
        if not p_times:
            return "time_missing_or_unparsed"
        if not g_times.issubset(p_times):
            return "time_mismatch"

    if set(row.get("predicted_evidence_turns", [])) & set(row.get("gold_evidence_turns", [])):
        return "answer_error_despite_some_gold_evidence"

    return "retrieval_or_reasoning_failure"


# ============================================================
# Main analysis
# ============================================================

def load_predictions(mode: str):
    path = OUTPUT_DIR / mode / "predictions.jsonl"

    if not path.exists():
        print(f"Missing: {path}")
        return []

    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    return rows


def analyze_mode(mode: str):
    rows = load_predictions(mode)

    if not rows:
        return None, None

    strict_correct = 0
    relaxed_correct = 0

    categorized = []
    category_counter = Counter()

    for row in rows:
        pred = row.get("predicted_answer", "")
        gold = row.get("gold_answer", "")

        strict = bool(row.get("answer_correct", False))
        relaxed = relaxed_match(pred, gold)

        if strict:
            strict_correct += 1

        if relaxed:
            relaxed_correct += 1

        category = "correct"
        if not strict:
            category = categorize_failure(pred, gold, row)
            category_counter[category] += 1

        new_row = dict(row)
        new_row["relaxed_answer_correct"] = relaxed
        new_row["failure_category"] = category
        categorized.append(new_row)

    n = len(rows)

    summary = {
        "mode": mode,
        "n_examples": n,
        "strict_accuracy": strict_correct / n,
        "relaxed_accuracy": relaxed_correct / n,
        "strict_bad": n - strict_correct,
        "relaxed_bad": n - relaxed_correct,
        "false_negative_recovered": relaxed_correct - strict_correct,
        "false_negative_recovered_rate": (relaxed_correct - strict_correct) / n,
    }

    # Save categorized predictions
    out_pred = OUT_ANALYSIS / f"{mode}_predictions_with_relaxed_eval.jsonl"

    with out_pred.open("w", encoding="utf-8") as f:
        for row in categorized:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    # Save category counts
    cat_rows = [
        {
            "mode": mode,
            "failure_category": k,
            "count": v,
            "rate_among_strict_failures": v / max(1, (n - strict_correct)),
            "rate_among_all": v / n,
        }
        for k, v in category_counter.most_common()
    ]

    cat_df = pd.DataFrame(cat_rows)
    cat_df.to_csv(
        OUT_ANALYSIS / f"{mode}_failure_categories.csv",
        index=False,
        encoding="utf-8",
    )

    return summary, cat_df


def main():
    summaries = []
    all_categories = []

    for mode in MODES:
        print("=" * 100)
        print("MODE:", mode)

        summary, cat_df = analyze_mode(mode)

        if summary is None:
            continue

        summaries.append(summary)

        print("Strict accuracy:", round(summary["strict_accuracy"], 4))
        print("Relaxed accuracy:", round(summary["relaxed_accuracy"], 4))
        print("Recovered false negatives:", summary["false_negative_recovered"])

        if cat_df is not None and not cat_df.empty:
            all_categories.append(cat_df)
            print("\nFailure categories:")
            print(cat_df.to_string(index=False))

    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(
        OUT_ANALYSIS / "relaxed_eval_summary.csv",
        index=False,
        encoding="utf-8",
    )

    if all_categories:
        all_cat_df = pd.concat(all_categories, ignore_index=True)
        all_cat_df.to_csv(
            OUT_ANALYSIS / "all_failure_categories.csv",
            index=False,
            encoding="utf-8",
        )

    print("\n" + "=" * 100)
    print("RELAXED EVAL SUMMARY")
    print("=" * 100)
    print(summary_df.to_string(index=False))

    print("\nSaved to:")
    print(OUT_ANALYSIS)


if __name__ == "__main__":
    main()

MODE: full_history
Strict accuracy: 0.5422
Relaxed accuracy: 0.9229
Recovered false negatives: 316

Failure categories:
        mode                        failure_category  count  rate_among_strict_failures  rate_among_all
full_history     false_negative_format_or_paraphrase    316                    0.831579        0.380723
full_history                   yes_no_semantic_error     33                    0.086842        0.039759
full_history answer_error_despite_some_gold_evidence     10                    0.026316        0.012048
full_history               duration_arithmetic_error      9                    0.023684        0.010843
full_history                date_missing_or_unparsed      6                    0.015789        0.007229
full_history                  date_or_calendar_error      3                    0.007895        0.003614
full_history                time_missing_or_unparsed      1                    0.002632        0.001205
full_history                           time_mism

In [2]:
import json
import re
from pathlib import Path

def norm(s):
    s = str(s).lower().strip()
    s = s.replace("p.m.", "pm").replace("a.m.", "am")
    s = s.replace("p.m", "pm").replace("a.m", "am")
    s = re.sub(r"\s+", " ", s)
    s = s.strip(" .,:;!?")
    return s

def extract_numbers(s):
    return re.findall(r"\d+", str(s).lower())

def relaxed_match(pred, gold):
    p = norm(pred)
    g = norm(gold)

    if p == g:
        return True

    if p in g or g in p:
        return True

    # yes/no matching
    yes_set = {"yes", "yeah", "correct", "true"}
    no_set = {"no", "not", "false"}

    if g in yes_set and p.startswith("yes"):
        return True
    if g in no_set and p.startswith("no"):
        return True

    # numeric/time overlap
    pn = extract_numbers(p)
    gn = extract_numbers(g)

    if gn and pn:
        # all gold numbers appear in prediction
        if all(x in pn for x in gn):
            return True

    return False

for mode in ["full_history", "semantic_rag", "timebound_rag", "oracle_evidence"]:
    path = Path(f"timebound_llm_outputs/{mode}/predictions.jsonl")
    rows = [json.loads(line) for line in path.open("r", encoding="utf-8")]

    strict = sum(x["answer_correct"] for x in rows) / len(rows)
    relaxed = sum(relaxed_match(x["predicted_answer"], x["gold_answer"]) for x in rows) / len(rows)

    print(mode)
    print("  strict:", round(strict, 4))
    print("  relaxed:", round(relaxed, 4))

full_history
  strict: 0.5422
  relaxed: 0.6867
semantic_rag
  strict: 0.5253
  relaxed: 0.6904
timebound_rag
  strict: 0.5289
  relaxed: 0.6916
oracle_evidence
  strict: 0.5301
  relaxed: 0.6843


In [4]:
import pandas as pd

df = pd.read_csv("timebound_llm_outputs/llm_by_task.csv")
df

,mode,task_type,n_examples,llm_success_rate,answer_accuracy,evidence_precision,evidence_recall,evidence_f1,contradiction_rate
0,full_history,aging_fact,99,1.0,0.545455,0.930976,0.895623,0.886532,0.565657
1,full_history,cancellation,104,1.0,0.480769,0.868590,0.770032,0.765064,0.740385
2,full_history,conflicting_updates,95,1.0,0.610526,0.933333,0.790351,0.817594,0.000000
3,full_history,delayed_observation,103,1.0,0.533981,0.936893,0.883495,0.884790,0.000000
4,full_history,elapsed_time_reasoning,98,1.0,0.663265,0.964286,0.931973,0.930272,0.000000
5,full_history,periodic_event,113,1.0,0.424779,0.889381,0.887906,0.858407,0.000000
6,full_history,rescheduling,98,1.0,0.418367,0.937075,0.766156,0.796259,0.051020
7,full_history,time_window_retrieval,120,1.0,0.658333,0.936111,0.869444,0.875278,0.025000
8,semantic_rag,aging_fact,99,1.0,0.545455,0.905724,0.905724,0.879798,0.545455
9,semantic_rag,cancellation,104,1.0,0.480769,0.886218,0.787660,0.795101,0.740385


# Long

In [1]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd


# ============================================================
# CONFIG
# ============================================================

OUTPUT_DIR = Path("timebound_long_llm_outputs")

MODES = [
    "full_history",
    "semantic_rag",
    "timebound_rag",
    "oracle_evidence",
]

OUT_ANALYSIS = OUTPUT_DIR / "failure_analysis"
OUT_ANALYSIS.mkdir(parents=True, exist_ok=True)


# ============================================================
# Normalization utilities
# ============================================================

MONTHS = {
    "jan": "january",
    "january": "january",
    "feb": "february",
    "february": "february",
    "mar": "march",
    "march": "march",
    "apr": "april",
    "april": "april",
    "may": "may",
    "jun": "june",
    "june": "june",
    "jul": "july",
    "july": "july",
    "aug": "august",
    "august": "august",
    "sep": "september",
    "sept": "september",
    "september": "september",
    "oct": "october",
    "october": "october",
    "nov": "november",
    "november": "november",
    "dec": "december",
    "december": "december",
}

MONTH_NUM = {
    "january": "01",
    "february": "02",
    "march": "03",
    "april": "04",
    "may": "05",
    "june": "06",
    "july": "07",
    "august": "08",
    "september": "09",
    "october": "10",
    "november": "11",
    "december": "12",
}


def norm_text(s):
    s = str(s).lower().strip()

    s = s.replace("a.m.", "am")
    s = s.replace("p.m.", "pm")
    s = s.replace("a.m", "am")
    s = s.replace("p.m", "pm")

    s = s.replace("cancelled", "canceled")

    # normalize month abbreviations
    for short, full in MONTHS.items():
        s = re.sub(rf"\b{short}\b", full, s)

    s = re.sub(r"\s+", " ", s)
    s = s.strip(" .,:;!?")

    return s


def extract_numbers(s):
    return re.findall(r"\d+", str(s))


def extract_yes_no(s):
    s = norm_text(s)

    if s.startswith("yes") or s in {"true", "correct", "valid", "active"}:
        return "yes"

    if (
        s.startswith("no")
        or s in {"false", "incorrect", "invalid", "inactive"}
        or "not valid" in s
        or "not active" in s
        or "not scheduled" in s
        or "canceled" in s
        or "expired" in s
        or "no future occurrence" in s
    ):
        return "no"

    return None


def extract_iso_datetimes(s):
    """
    Extract normalized ISO datetimes:
      YYYY-MM-DDTHH:MM
      YYYY-MM-DD
    """
    s = str(s)
    out = set()

    # datetime
    for m in re.finditer(r"\b(\d{4})-(\d{2})-(\d{2})[T\s](\d{2}):(\d{2})(?::\d{2})?\b", s):
        y, mo, d, h, mi = m.groups()
        out.add(f"{y}-{mo}-{d}T{h}:{mi}")

    # date only
    for m in re.finditer(r"\b(\d{4})-(\d{2})-(\d{2})\b", s):
        y, mo, d = m.groups()
        out.add(f"{y}-{mo}-{d}")

    return out


def extract_natural_dates(s):
    """
    Extract:
      August 12, 2026
      August 12 2026
      August 12
    If year is missing, returns month-day only.
    """
    s = norm_text(s)
    out = set()

    # Month day year
    pattern1 = (
        r"\b(january|february|march|april|may|june|july|august|september|october|november|december)"
        r"\s+(\d{1,2})(?:st|nd|rd|th)?(?:,)?\s+(\d{3,4})\b"
    )

    for m in re.finditer(pattern1, s):
        month, day, year = m.groups()
        out.add(f"{year}-{MONTH_NUM[month]}-{int(day):02d}")

    # Month day without year
    pattern2 = (
        r"\b(january|february|march|april|may|june|july|august|september|october|november|december)"
        r"\s+(\d{1,2})(?:st|nd|rd|th)?\b"
    )

    for m in re.finditer(pattern2, s):
        month, day = m.groups()
        out.add(f"XXXX-{MONTH_NUM[month]}-{int(day):02d}")

    return out


def extract_month_year(s):
    s = norm_text(s)
    out = set()

    pattern = (
        r"\b(january|february|march|april|may|june|july|august|september|october|november|december)"
        r"(?:,)?\s+(\d{3,4})\b"
    )

    for m in re.finditer(pattern, s):
        month, year = m.groups()
        out.add((month, year))

    return out


def extract_times(s):
    s = norm_text(s)
    out = set()

    # HH:MM
    for h, m in re.findall(r"\b(\d{1,2}):(\d{2})\b", s):
        hh = int(h)
        mm = int(m)
        if 0 <= hh <= 23 and 0 <= mm <= 59:
            out.add(f"{hh:02d}:{mm:02d}")

    # 2 pm / 10 AM
    for h, ampm in re.findall(r"\b(\d{1,2})\s*(am|pm)\b", s):
        hh = int(h)

        if ampm == "pm" and hh != 12:
            hh += 12

        if ampm == "am" and hh == 12:
            hh = 0

        if 0 <= hh <= 23:
            out.add(f"{hh:02d}:00")

    return out


def normalize_duration_hours(s):
    """
    Extract duration in hours from:
      56 hours
      5 days and 21 hours
      2 days 8 hours
    """
    s = norm_text(s)

    d = re.search(r"\b(\d+)\s*days?\b", s)
    h = re.search(r"\b(\d+)\s*hours?\b", s)

    if d:
        days = int(d.group(1))
        hours = int(h.group(1)) if h else 0
        return days * 24 + hours

    h2 = re.search(r"\b(\d+)\s*hours?\b", s)
    if h2:
        return int(h2.group(1))

    return None


def date_sets_match(pred_dates, gold_dates):
    """
    Allows:
      2026-08-12T14:00 matching 2026-08-12 if gold only asks date.
      natural date without year matching same month-day when gold also has same month-day.
    """
    if not gold_dates:
        return False

    if not pred_dates:
        return False

    for gd in gold_dates:
        matched = False

        gd_date = gd.split("T")[0]

        for pd in pred_dates:
            pd_date = pd.split("T")[0]

            if gd == pd:
                matched = True
                break

            if gd_date == pd_date:
                matched = True
                break

            # Month-day match when one side has XXXX year.
            if gd_date.startswith("XXXX-"):
                if pd_date.endswith(gd_date[4:]):
                    matched = True
                    break

            if pd_date.startswith("XXXX-"):
                if gd_date.endswith(pd_date[4:]):
                    matched = True
                    break

        if not matched:
            return False

    return True


def relaxed_match(pred, gold):
    p = norm_text(pred)
    g = norm_text(gold)

    if not g:
        return False

    if p == g:
        return True

    if p in g or g in p:
        return True

    # yes/no
    py = extract_yes_no(pred)
    gy = extract_yes_no(gold)

    if py is not None and gy is not None and py == gy:
        return True

    # durations
    ph = normalize_duration_hours(pred)
    gh = normalize_duration_hours(gold)

    if ph is not None and gh is not None and ph == gh:
        return True

    # ISO + natural dates
    p_dates = extract_iso_datetimes(pred) | extract_natural_dates(pred)
    g_dates = extract_iso_datetimes(gold) | extract_natural_dates(gold)

    if date_sets_match(p_dates, g_dates):
        return True

    # month-year, for TempReason-like answers
    p_my = extract_month_year(pred)
    g_my = extract_month_year(gold)

    if g_my and p_my and g_my.issubset(p_my):
        return True

    # time only
    p_times = extract_times(pred)
    g_times = extract_times(gold)

    if g_times and p_times and g_times.issubset(p_times):
        return True

    # numeric fallback
    pn = extract_numbers(pred)
    gn = extract_numbers(gold)

    if gn and pn and all(x in pn for x in gn):
        return True

    return False


# ============================================================
# Failure categories
# ============================================================

def categorize_failure(pred, gold, row):
    if relaxed_match(pred, gold):
        return "false_negative_format_or_paraphrase"

    py = extract_yes_no(pred)
    gy = extract_yes_no(gold)

    if py is not None or gy is not None:
        if py != gy:
            return "yes_no_semantic_error"
        return "yes_no_format_issue"

    ph = normalize_duration_hours(pred)
    gh = normalize_duration_hours(gold)

    if gh is not None:
        if ph is None:
            return "duration_missing_or_unparsed"
        if ph != gh:
            return "duration_arithmetic_error"

    p_dates = extract_iso_datetimes(pred) | extract_natural_dates(pred)
    g_dates = extract_iso_datetimes(gold) | extract_natural_dates(gold)

    if g_dates:
        if not p_dates:
            return "date_missing_or_unparsed"
        if not date_sets_match(p_dates, g_dates):
            return "date_or_calendar_error"

    p_times = extract_times(pred)
    g_times = extract_times(gold)

    if g_times:
        if not p_times:
            return "time_missing_or_unparsed"
        if not g_times.issubset(p_times):
            return "time_mismatch"

    pred_evidence = set(row.get("predicted_evidence_turns", []))
    gold_evidence = set(row.get("gold_evidence_turns", []))

    if pred_evidence & gold_evidence:
        return "answer_error_despite_some_gold_evidence"

    return "retrieval_or_reasoning_failure"


# ============================================================
# IO and analysis
# ============================================================

def load_predictions(mode):
    path = OUTPUT_DIR / mode / "predictions.jsonl"

    if not path.exists():
        print(f"Missing: {path}")
        return []

    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))

    return rows


def analyze_mode(mode):
    rows = load_predictions(mode)

    if not rows:
        return None, None

    strict_correct = 0
    relaxed_correct = 0

    categorized = []
    category_counter = Counter()

    for row in rows:
        pred = row.get("predicted_answer", "")
        gold = row.get("gold_answer", "")

        strict = bool(row.get("answer_correct", False))
        relaxed = relaxed_match(pred, gold)

        if strict:
            strict_correct += 1

        if relaxed:
            relaxed_correct += 1

        if strict:
            category = "correct"
        else:
            category = categorize_failure(pred, gold, row)
            category_counter[category] += 1

        new_row = dict(row)
        new_row["relaxed_answer_correct"] = relaxed
        new_row["failure_category"] = category
        categorized.append(new_row)

    n = len(rows)

    summary = {
        "mode": mode,
        "n_examples": n,
        "strict_accuracy": strict_correct / n,
        "relaxed_accuracy": relaxed_correct / n,
        "strict_bad": n - strict_correct,
        "relaxed_bad": n - relaxed_correct,
        "false_negative_recovered": relaxed_correct - strict_correct,
        "false_negative_recovered_rate": (relaxed_correct - strict_correct) / n,
    }

    out_pred = OUT_ANALYSIS / f"{mode}_predictions_with_relaxed_eval.jsonl"

    with out_pred.open("w", encoding="utf-8") as f:
        for row in categorized:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    cat_rows = [
        {
            "mode": mode,
            "failure_category": k,
            "count": v,
            "rate_among_strict_failures": v / max(1, (n - strict_correct)),
            "rate_among_all": v / n,
        }
        for k, v in category_counter.most_common()
    ]

    cat_df = pd.DataFrame(cat_rows)
    cat_df.to_csv(
        OUT_ANALYSIS / f"{mode}_failure_categories.csv",
        index=False,
        encoding="utf-8",
    )

    return summary, cat_df


def main():
    summaries = []
    all_categories = []

    print("=" * 100)
    print("Relaxed evaluation for TimeBound-Long")
    print("=" * 100)
    print(f"Input dir: {OUTPUT_DIR}")
    print(f"Output dir: {OUT_ANALYSIS}")

    for mode in MODES:
        print("\n" + "=" * 100)
        print("MODE:", mode)

        summary, cat_df = analyze_mode(mode)

        if summary is None:
            continue

        summaries.append(summary)

        print("Strict accuracy:", round(summary["strict_accuracy"], 4))
        print("Relaxed accuracy:", round(summary["relaxed_accuracy"], 4))
        print("Recovered false negatives:", summary["false_negative_recovered"])

        if cat_df is not None and not cat_df.empty:
            all_categories.append(cat_df)
            print("\nFailure categories:")
            print(cat_df.to_string(index=False))

    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(
        OUT_ANALYSIS / "relaxed_eval_summary.csv",
        index=False,
        encoding="utf-8",
    )

    if all_categories:
        all_cat_df = pd.concat(all_categories, ignore_index=True)
        all_cat_df.to_csv(
            OUT_ANALYSIS / "all_failure_categories.csv",
            index=False,
            encoding="utf-8",
        )

    print("\n" + "=" * 100)
    print("RELAXED EVAL SUMMARY")
    print("=" * 100)

    if not summary_df.empty:
        print(summary_df.to_string(index=False))

    print("\nSaved to:")
    print(OUT_ANALYSIS)


if __name__ == "__main__":
    main()

Relaxed evaluation for TimeBound-Long
Input dir: timebound_long_llm_outputs
Output dir: timebound_long_llm_outputs\failure_analysis

MODE: full_history
Strict accuracy: 0.303
Relaxed accuracy: 0.614
Recovered false negatives: 311

Failure categories:
        mode                        failure_category  count  rate_among_strict_failures  rate_among_all
full_history     false_negative_format_or_paraphrase    311                    0.446198           0.311
full_history                  date_or_calendar_error    157                    0.225251           0.157
full_history               duration_arithmetic_error    118                    0.169297           0.118
full_history                   yes_no_semantic_error     76                    0.109039           0.076
full_history          retrieval_or_reasoning_failure     32                    0.045911           0.032
full_history answer_error_despite_some_gold_evidence      3                    0.004304           0.003

MODE: semantic_rag
S

In [2]:
import pandas as pd

df = pd.read_csv("timebound_long_llm_outputs/llm_by_task.csv")
df

,mode,task_type,n_examples,llm_success_rate,answer_accuracy,evidence_precision,evidence_recall,evidence_f1,contradiction_rate
0,full_history,aging_fact,125,1.0,0.328,0.741800,0.800000,0.707644,0.472
1,full_history,cancellation,125,1.0,0.000,0.769143,0.884000,0.800279,0.160
2,full_history,conflicting_updates,125,1.0,0.216,0.212000,0.072000,0.107200,0.000
3,full_history,delayed_observation,125,1.0,0.608,0.438533,0.560000,0.471886,0.000
4,full_history,elapsed_time_reasoning,125,1.0,0.040,0.779514,0.997333,0.855823,0.000
5,full_history,periodic_event,125,1.0,0.104,0.451721,0.768000,0.555093,0.000
6,full_history,rescheduling,125,1.0,0.520,0.540267,0.184000,0.272667,0.000
7,full_history,time_window_retrieval,125,1.0,0.608,0.583733,0.424000,0.461410,0.000
8,semantic_rag,aging_fact,125,1.0,0.168,0.238667,0.296000,0.245600,0.848
9,semantic_rag,cancellation,125,1.0,0.000,0.800000,0.456000,0.566400,0.136


In [3]:
import json
import pandas as pd
from pathlib import Path

base = Path("timebound_long_llm_outputs/failure_analysis")

modes = [
    "full_history",
    "semantic_rag",
    "timebound_rag",
    "oracle_evidence",
]

rows = []

for mode in modes:
    path = base / f"{mode}_predictions_with_relaxed_eval.jsonl"
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            rows.append({
                "mode": mode,
                "task_type": obj["task_type"],
                "strict_correct": bool(obj.get("answer_correct", False)),
                "relaxed_correct": bool(obj.get("relaxed_answer_correct", False)),
                "evidence_f1": float(obj.get("evidence_f1", 0.0)),
                "contradiction": bool(obj.get("contradiction", False)),
            })

df = pd.DataFrame(rows)

summary = (
    df.groupby(["mode", "task_type"])
    .agg(
        n_examples=("relaxed_correct", "size"),
        strict_accuracy=("strict_correct", "mean"),
        relaxed_accuracy=("relaxed_correct", "mean"),
        evidence_f1=("evidence_f1", "mean"),
        contradiction_rate=("contradiction", "mean"),
    )
    .reset_index()
)

out = Path("timebound_long_llm_outputs/failure_analysis/relaxed_by_task.csv")
summary.to_csv(out, index=False, encoding="utf-8")

print(summary.to_string(index=False))
print("\nSaved:", out)

           mode              task_type  n_examples  strict_accuracy  relaxed_accuracy  evidence_f1  contradiction_rate
   full_history             aging_fact         125            0.328             0.712     0.707644               0.472
   full_history           cancellation         125            0.000             0.936     0.800279               0.160
   full_history    conflicting_updates         125            0.216             0.376     0.107200               0.000
   full_history    delayed_observation         125            0.608             0.648     0.471886               0.000
   full_history elapsed_time_reasoning         125            0.040             0.056     0.855823               0.000
   full_history         periodic_event         125            0.104             0.888     0.555093               0.000
   full_history           rescheduling         125            0.520             0.688     0.272667               0.000
   full_history  time_window_retrieval         1